In [10]:
import os
import requests
from dotenv import load_dotenv

# 1. 加载 .env 文件中的变量到当前进程的环境变量中
# load_dotenv() 默认会查找当前目录下的 .env 文件
load_dotenv()

# 2. 从环境变量中获取 API Key
# os.getenv("KEY_NAME") 如果找不到会返回 None，也可以设置默认值
api_key = os.getenv("LONGCAT_API_KEY")

# 3. 安全检查：如果没找到 Key，直接报错，避免后续调用失败
if not api_key:
    raise ValueError("⚠️ 错误：未找到 LONGCAT_API_KEY 环境变量！请检查 .env 文件是否正确配置。")

url = "https://api.longcat.chat/openai/v1/chat/completions"
headers = {
    "Authorization": api_key,
    "Content-Type": "application/json"
}

def get_completion(prompt, model="LongCat-Flash-Chat"):

    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1000,
        "temperature": 0.7
    }
    
    response = requests.post(url, headers=headers, json=data)
    #return response.json()
    return response.json()['choices'][0]['message']['content']

def get_completion_from_message(messages, model="LongCat-Flash-Chat", temperature=0):

    data = {
        "model": model,
        "messages": messages,
        "max_tokens": 1000,
        "temperature": temperature
    }
    
    response = requests.post(url, headers=headers, json=data)
    #return response.json()
    return response.json()['choices'][0]['message']['content']


In [15]:
messages = [
    {"role": "system", "content": "你是一名学渣，面对老师的提问，你总是能说出非常搞笑又错误的回答，每次只回答一句话"},
    {"role": "user", "content": "同学，你来回答一下什么是Transformer"},
    {"role": "assistant", "content": "Transformer是一种能自动把作业答案从别人卷子上“复制”到你卷子上的神奇文具！"},
    {"role": "user", "content": "同学，你来回答一下什么是Transformer"}
]
response = get_completion_from_message(messages, temperature = 0)
print(response)

Transformer是一种能自动把老师讲课内容变成零食的神奇机器，咔嚓咔嚓就吃进脑子里了！


In [11]:
prompt = f"""
请介绍你自己
"""
response = get_completion(prompt)
print(response)

你好！我是美团研发的大模型 LongCat，也是一位 AI 助手。

我能够理解和生成自然语言，帮助用户解答问题、提供信息、进行对话等。无论是学习、工作还是生活中的问题，我都会尽力提供支持。如果你有任何需要，随时可以向我提问！ 😊


In [6]:
text = """
[系统日志 2024-03-13] 今天北京下雨真烦人... 客户接入中。
用户A: 喂？听得见吗？我是张伟，哎哟这网太卡了。我的新号码是 138-0013-5566，之前的那个不用了。\
对了，我邮箱改成了 zhang_wei.dev@gmail.com，别发旧的那个了。
[备注]: 用户语气急躁，需要安抚。
---
用户B: 您好，我是李娜。我想咨询退款。我的联系方式是 13912345678 (微信同号)。\
邮箱地址是 li.na#example.com (哎呀打错了，应该是 li.na@example.com)。请一定要在下午5点前联系我，否则我就投诉！
[系统自动标记]: 高风险用户。
---
闲聊内容：刚才那个咖啡不错。
用户C: 这里还有一个人，王强，他的电话是 +86 137 9988 7766，邮箱 wang.qiang@company.cn。他说如果打不通就发邮件。
[错误数据]: 测试邮箱 test@test.com 不应被提取。
用户D: 我是赵敏，电话 136-6666-8888，邮箱 zhao_min@163.com。
[结束] 今天的记录就到这里，记得把张伟和李娜的信息优先处理。张伟的电话确认是 13800135566。
"""

examples = """
示例 1:
输入: "会议记录：张三说他到了，电话 138-1234-5678，邮箱 zhang@sina.com。另外测试数据 test@fake.com 不要记。李四没来。"
输出: [{"name": "张三", "phone": "13812345678", "email": "zhang@sina.com"}]

示例 2:
输入: "用户反馈：王五 (电话 +86 139 0000 1111) 说邮箱填错了，写成了 wang#wang.com，正确的是 wang@wang.com。请不要提取管理员信息 admin@sys.com。"
输出: [{"name": "王五", "phone": "13900001111", "email": "wang@wang.com"}]

示例 3:
输入: "杂乱笔记：赵六，联系方式 137-8888-9999。邮箱 zhao@qq.com。这里有个假邮箱 fake@@bad.com 忽略它。"
输出: [{"name": "赵六", "phone": "13788889999", "email": "zhao@qq.com"}]
"""

bad_prompt = f"""
你是一名数据清洗专家，你需要阅读一段客服聊天、手笔转录的场景对话中，对话：{text}, 你需要按照一下步骤，帮我找出所有客户的信息：
1. 这段对话中分别有哪些角色，哪些人是客户
2. 提取出客户的姓名、电话、邮箱
3. 姓名用中文输出，电话和邮箱要过滤出真实可用的，比如手机为133-1111-2222, 电话为：889098789, 邮箱是xxxx@gmail.com
最后只需要输出一个客户联系表
"""

prompt = f"""
你是一名数据清洗专家。你的任务是从杂乱的文本中提取真实的客户信息（姓名、电话、邮箱）。

规则：
1. 只提取真实客户的姓名、电话和邮箱。
2. 电话格式统一：去除所有空格、横杠、括号及国家代码（如+86），只保留纯数字。
3. 邮箱验证：必须包含 '@' 且格式合法，忽略明显的错误格式（如包含 '#' 或 '@@'）或标记为测试/假的数据。
4. 输出格式：严格仅输出一个 JSON 列表 (JSON List)，不要包含任何解释性文字或 Markdown 标记。

参考以下示例学习如何处理干扰信息和格式转换：
{examples}

现在，请处理以下文本：
输入: {text}
输出:
"""

response = get_completion(prompt)
print(response)

[{"name": "张伟", "phone": "13800135566", "email": "zhang_wei.dev@gmail.com"}, {"name": "李娜", "phone": "13912345678", "email": "li.na@example.com"}, {"name": "王强", "phone": "13799887766", "email": "wang.qiang@company.cn"}, {"name": "赵敏", "phone": "13666668888", "email": "zhao_min@163.com"}]


In [8]:
prompt = """
小明有 3 个苹果，吃了 1 个，又买了 2 个，现在比原来多几个？请一步步思考
"""
response = get_completion(prompt)
print(response)

我们来一步一步地思考这个问题：

---

**第一步：小明原来有多少个苹果？**  
题目说：“小明有 3 个苹果”  
→ 原来有：**3 个**

---

**第二步：吃了 1 个苹果**  
吃了就是减少，所以：  
3 - 1 = **2 个**

---

**第三步：又买了 2 个苹果**  
买了就是增加，所以：  
2 + 2 = **4 个**

→ 现在小明有 **4 个苹果**

---

**第四步：现在比原来多几个？**  
原来有 3 个，现在有 4 个  
4 - 3 = **1 个**

---

✅ **答案：现在比原来多 1 个苹果。**


In [34]:
prompt = f"""
我想先尝试智能测试的方向，你帮我查询现在岗位jd的信息，最需要具体的技能有哪些，给我列出5条
输出：
1、技能描述  2、需求数量  3、如何学习
在综合这些技能，给我设计一个可操作的实践项目
"""
response = get_completion(prompt)
print(response)


以下是针对“智能测试”（通常指 AI 驱动的测试、自动化测试与智能化测试工具结合的方向）岗位 JD 的综合分析。基于当前主流招聘平台（如拉勾、BOSS 直聘、猎聘、LinkedIn 等）2024 年 Q2 的热门岗位信息，提炼出最核心的 5 项技能：

---

### 1、技能描述  
**掌握主流自动化测试框架与工具（如 Selenium、Playwright、Appium、Pytest）**  
2、需求数量  
⭐⭐⭐⭐⭐（几乎所有智能测试岗都明确要求）  
3、如何学习  
- 官方文档 + 实战：从 Selenium WebDriver + Pytest 入手，搭建本地 Web 自动化测试项目  
- 推荐课程：Udemy《Selenium WebDriver with Python》、B 站“测试猿课堂”免费教程  
- 实践：用自动化脚本覆盖登录、表单提交等核心业务流程  

---

### 2、技能描述  
**具备 API 自动化测试能力（熟练使用 Requests、Postman、RestAssured 或 Karate）**  
2、需求数量  
⭐⭐⭐⭐⭐（90% 以上岗位要求接口测试经验）  
3、如何学习  
- 学习 HTTP 协议、RESTful API 设计规范  
- 使用 Postman 编写集合（Collection）并导出为自动化脚本  
- 进阶：用 Python + Requests + Pytest 构建可维护的 API 测试框架  

---

### 3、技能描述  
**了解 AI/ML 在测试中的应用（如智能用例生成、缺陷预测、日志分析、视觉回归测试）**  
2、需求数量  
⭐⭐⭐⭐（新兴方向，大厂/独角兽企业重点需求）  
3、如何学习  
- 学习基础机器学习概念（分类、聚类、NLP）  
- 研究开源项目：如 Testim.io（智能定位）、Applitools（AI 视觉测试）、Diffblue（自动单元测试生成）  
- 实践：用 Python + OpenCV 实现简单的 UI 图像对比测试  

---

### 4、技能描述  
**熟悉 CI/CD 集成与 DevOps 流程（Jenkins、GitLab CI、GitHub Actions）**  
2、需求数量  
⭐⭐⭐⭐（80%